# ML: подготовка данных для прогноза нагрузки на следующие сутки

Первый шаг этапа 2 из [бизнес-задачи](../BUSINESS_TASK.md): получить общий ряд с шагом 15 минут, проверить его и разделить по времени. Модели и оценки точности здесь ещё не рассчитываются.

**Результат:** `total_load`, `train`, `validation`, `test` и локальный CSV в `data/ml/total_load_15min.csv`. Исходный файл и EDA-ноутбук не изменяются.

## 1. Настройка и параметры

Используйте зависимости из `requirements.txt` и Python 3.14. В облачной среде выберите ядро **ElectricityLoad (Python 3.14)**; локально — ядро вашего виртуального окружения.

Источник: [UCI ElectricityLoadDiagrams20112014](https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014). До запуска разместите оригинальный `LD2011_2014.txt` в `data/`. Скачать ZIP можно с [официального источника](https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip); сам ноутбук не обращается к сети.

Рабочий горизонт — **96 интервалов (сутки)**. Планируем прогнозировать среднюю мощность интервала, кВт; энергия интервала равна мощности × 0,25 часа.

In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "data_load.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Запустите ноутбук из корня проекта или папки notebooks.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

INTERVAL_MINUTES = 15
INTERVAL_HOURS = INTERVAL_MINUTES / 60
HORIZON = 96
VALIDATION_START = pd.Timestamp("2014-01-01")
TEST_START = pd.Timestamp("2014-07-01")
DATA_PATH = project_root / "data" / "LD2011_2014.txt"
OUTPUT_PATH = project_root / "data" / "ml" / "total_load_15min.csv"

print(f"Python: {sys.version.split()[0]}; pandas: {pd.__version__}; numpy: {np.__version__}")

Python: 3.14.7; pandas: 2.3.3; numpy: 2.5.3


## 2. Загрузка оригинальных данных

Читаем все клиентские столбцы: разделитель `;`, десятичная запятая, первая колонка — временная метка конца интервала. Размер исходного файла около 711 МБ; для загрузки и проверок нужно несколько гигабайт памяти. Контрольная сумма фиксирует именно использованный файл, но не заменяет проверку происхождения.

In [2]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Не найден {DATA_PATH}. Скачайте оригинальный датасет UCI в data/.")

with DATA_PATH.open("rb") as source:
    source_sha256 = hashlib.file_digest(source, "sha256").hexdigest()

raw_data = pd.read_csv(
    DATA_PATH, sep=";", decimal=",", parse_dates=[0], index_col=0,
    encoding="utf-8", low_memory=False,
)
raw_data.index.name = "interval_end"
display(pd.Series({
    "rows": len(raw_data), "clients": len(raw_data.columns),
    "first_interval_end": raw_data.index.min(),
    "last_interval_end": raw_data.index.max(),
    "source_sha256": source_sha256,
}, name="source"))

rows                                                             140256
clients                                                             370
first_interval_end                                  2011-01-01 00:15:00
last_interval_end                                   2015-01-01 00:00:00
source_sha256         d51565f2cb5a6b768d06ba1bbd3c084c6e2f3aab07f00c...
Name: source, dtype: object

## 3. Проверки качества

Проверки останавливают выполнение при нарушении структуры: пропуски, бесконечности, отрицательные значения, дубликаты или разрывы сетки. Мы не исправляем эти проблемы автоматически.

**Нули сохраняем:** они могут отражать начало истории клиента или особенности перевода часов. Метки остаются в исходном локальном времени без назначения UTC: набор имеет специальное представление переходов часов. Разный период активности клиентов означает меняющийся состав общей нагрузки — это нужно учитывать при оценке будущего прогноза.

In [3]:
assert isinstance(raw_data.index, pd.DatetimeIndex), "Некорректный тип временного индекса"
assert len(raw_data) > HORIZON and len(raw_data.columns) > 0, "Недостаточно данных"
assert not raw_data.index.hasnans, "Не распознаны временные метки"
assert raw_data.index.is_monotonic_increasing, "Время не отсортировано"
assert raw_data.index.is_unique, "Есть повторяющиеся временные метки"
assert raw_data.columns.is_unique, "Есть повторяющиеся имена клиентов"
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in raw_data.dtypes), "Нечисловые столбцы"
assert raw_data.index.to_series().diff().iloc[1:].dt.total_seconds().eq(INTERVAL_MINUTES * 60).all(), "Нарушена сетка 15 минут"
values = raw_data.to_numpy()
assert np.isfinite(values).all(), "Есть NaN или бесконечные значения"
assert (values >= 0).all(), "Есть отрицательная мощность"

display(pd.Series({
    "missing_values": int(raw_data.isna().sum().sum()),
    "duplicate_timestamps": int(raw_data.index.duplicated().sum()),
    "zero_values": int((values == 0).sum()),
    "all_zero_intervals": int((values == 0).all(axis=1).sum()),
}, name="quality"))
print("Проверки исходных данных пройдены.")

missing_values                 0
duplicate_timestamps           0
zero_values             10457342
all_zero_intervals             0
Name: quality, dtype: int64

Проверки исходных данных пройдены.


## 4. Общий ряд нагрузки

Суммируем мощность **всех** клиентов, без исключения клиентов по правилам кластеризации. Переносим метку конца интервала на его начало: `00:15` становится `00:00`. Так каждый день состоит из 96 интервалов.

`total_power_kw` — будущая целевая величина; `total_energy_kwh` — та же нагрузка в единицах энергии, а не независимый прогнозный признак.

In [4]:
from src.eda_utils import get_total_load, shift_to_interval_start

total_load = shift_to_interval_start(get_total_load(raw_data, INTERVAL_HOURS), INTERVAL_MINUTES)
total_load.index.name = "timestamp"

np.testing.assert_allclose(total_load["total_power_kw"].to_numpy(), values.sum(axis=1))
np.testing.assert_allclose(
    total_load["total_energy_kwh"].to_numpy(),
    total_load["total_power_kw"].to_numpy() * INTERVAL_HOURS,
)
assert total_load.index.is_unique and total_load.index.is_monotonic_increasing
assert total_load.index[0] == total_load.index[0].normalize(), "Неполный первый день"
assert total_load.index[-1].time() == pd.Timestamp("2000-01-01 23:45:00").time(), "Неполный последний день"
assert total_load.groupby(total_load.index.normalize()).size().eq(HORIZON).all(), "Есть неполные сутки"
display(total_load.head())

,total_power_kw,total_energy_kwh
timestamp,,
2011-01-01 00:00:00,68513.115338,17128.278835
2011-01-01 00:15:00,69180.304361,17295.076090
2011-01-01 00:30:00,69364.850573,17341.212643
2011-01-01 00:45:00,68350.480660,17087.620165
2011-01-01 01:00:00,66166.874303,16541.718576


## 5. Разделение только по времени

- Обучение: до 1 января 2014 года.
- Валидация: январь–июнь 2014 года.
- Тест: июль–декабрь 2014 года.

Границы заданы заранее и проходят по началу суток. Случайное перемешивание не используем. Гиперпараметры будем выбирать на валидации; тест оставим для итоговой оценки. Для суточного прогноза будущие признаки должны быть известны в момент начала суток: нельзя подставлять фактическую нагрузку уже прогнозируемых интервалов.

In [5]:
assert VALIDATION_START < TEST_START
assert VALIDATION_START == VALIDATION_START.normalize()
assert TEST_START == TEST_START.normalize()

train = total_load.loc[total_load.index < VALIDATION_START].copy()
validation = total_load.loc[(total_load.index >= VALIDATION_START) & (total_load.index < TEST_START)].copy()
test = total_load.loc[total_load.index >= TEST_START].copy()

splits = {"train": train, "validation": validation, "test": test}
for name, frame in splits.items():
    assert not frame.empty, f"Пустая выборка: {name}"
    assert len(frame) % HORIZON == 0, f"Неполные сутки: {name}"
assert train.index.max() < validation.index.min() <= validation.index.max() < test.index.min()
pd.testing.assert_frame_equal(pd.concat(splits.values()), total_load)

split_summary = pd.DataFrame([
    {"split": name, "start": frame.index.min(), "end": frame.index.max(),
     "intervals": len(frame), "days": len(frame) // HORIZON}
    for name, frame in splits.items()
]).set_index("split")
display(split_summary)

,start,end,intervals,days
split,,,,
train,2011-01-01,2013-12-31 23:45:00,105216,1096
validation,2014-01-01,2014-06-30 23:45:00,17376,181
test,2014-07-01,2014-12-31 23:45:00,17664,184


## 6. Сохранение подготовленных данных

Экспортируем общий ряд с колонкой `split`. Файл создаётся в игнорируемой Git папке `data/ml/`; при повторном выполнении обновляется только этот производный CSV. Он и исходный датасет не отправляются в GitHub. Ноутбук содержит код и компактные результаты проверок.

In [6]:
prepared_data = pd.concat(
    [frame.assign(split=name) for name, frame in splits.items()]
)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
prepared_data.to_csv(OUTPUT_PATH, index=True)

restored = pd.read_csv(OUTPUT_PATH, parse_dates=["timestamp"], index_col="timestamp")
pd.testing.assert_frame_equal(restored, prepared_data, check_freq=False)
print(f"Сохранено: data/ml/{OUTPUT_PATH.name}")
print(f"Интервалов: {len(prepared_data):,}; клиентов в исходной сумме: {len(raw_data.columns)}")
print("Проверки агрегации, полных суток, разделения и повторного чтения CSV пройдены.")

Сохранено: data/ml/total_load_15min.csv
Интервалов: 140,256; клиентов в исходной сумме: 370
Проверки агрегации, полных суток, разделения и повторного чтения CSV пройдены.


## Следующий шаг

На одинаковых суточных точках старта сравнить три базовых прогноза на 96 интервалов: повтор последнего известного значения, профиль предыдущих суток и профиль недели назад. Рассчитать MAE, RMSE и WAPE на валидации, затем добавить одну простую ML-модель.

На этом этапе подготовлены данные и временные выборки. Прогнозы ещё не построены; точность и полезность модели не оценивались. CSV не содержит лагов, обученных преобразований или будущих целевых векторов.